# 🐝🔌 Measuring power consumption on Xylo™Audio HDK

> **_Note:_** In this tutorial Xylo™Audio refers to Xylo™Audio 2 and Xylo™Audio 3. In case differences occur we will explicitly name the correct HDK.

> **_Note 2:_** This tutorial assume you have already build and mapped a network to Xylo™Audio. If this is not the case, please follow the tutorial at :ref:`/devices/quick-xylo/xylo-audio-intro.ipynb`

In [ ]:
# Specify the xylo board you would like to use for this tutorial
# Available boards: XyloAudio2 and XyloAudio3
xylo_board_name = 'XyloAudio3'

# - Imports dependent on your HDK
# - XyloAudio 2
if xylo_board_name == 'XyloAudio2':
    import rockpool.devices.xylo.syns61201 as xa
    from rockpool.devices.xylo.syns61201 import xa2_devkit_utils as xa_utils
# - XyloAudio 3
elif xylo_board_name == 'XyloAudio3':
    import rockpool.devices.xylo.syns65302 as xa
    from rockpool.devices.xylo.syns65302 import xa3_devkit_utils as xa_utils

# Available boards: XyloAudio2 and XyloAudio3
xylo_board_name = 'XyloAudio3'


import numpy as np

In [2]:
# - Find and connect to a XyloAudio HDK
from rockpool.devices.xylo import find_xylo_hdks
xylo_hdk_nodes, modules, versions = find_xylo_hdks()
print(xylo_hdk_nodes)

hdk = None

for version, xylo in zip(versions, xylo_hdk_nodes):
    if version == "syns61201":
        hdk = xylo
    # - For XyloAudio 3
    elif version == "syns65302":
        hdk = xylo

if hdk is None:
    assert False, 'This tutorial requires a connected XyloAudio HDK to demonstrate.'




The connected Xylo HDK contains a XyloAudio 3. Importing `rockpool.devices.xylo.syns65302`


In [3]:
# - Define the size of the network layers
Nin = 2
Nhidden = 4
Nout = 2
dt = 1e-3

# load config created in XyloAudio intro tutorial
%store -r config

# - For XyloAudio 2
if xylo_board_name == 'XyloAudio2':
    # - Set a low clock frequency for the XyloAudio 2 device
    xa_utils.set_xylo_core_clock_freq(hdk, 6.25)
    # - Use XyloSamna to deploy to the HDK
    modSamna = xa.XyloSamna(hdk, config, dt = 10e-3, power_frequency=20.)
# - For XyloAudio 3
elif xylo_board_name == 'XyloAudio3':
    import samna
    # - Set a low clock frequency for the XyloAudio 3 device
    xa_utils.set_xylo_core_clock_freq(hdk, 6.25)
    # - Use XyloSamna to deploy to the HDK
    modSamna = xa.XyloSamna(hdk, config, dt = 10e-3, power_frequency=20.)
    
print(modSamna)

XyloSamna  with shape (2, 8, 2)


In [4]:
# - Generate some Poisson input
T = 1000
f = 0.4
input_spikes = np.random.rand(T, Nin) < f

# - Evolve some input on the SNN core, and record power during inference
out, _, record_dict = modSamna(input_spikes, record_power = True)

print(record_dict)

{'io_power': array([4.46644719e-05, 5.27330065e-05, 5.42548444e-05, 5.26624547e-05,
       5.22731397e-05, 5.24438111e-05, 5.15508297e-05, 5.20056725e-05,
       5.31936278e-05, 5.20105327e-05, 5.31826708e-05, 5.24881628e-05,
       5.22707500e-05, 5.29561894e-05, 5.17823424e-05, 5.23655495e-05,
       5.30992342e-05, 5.22785888e-05, 5.24720042e-05, 5.34786841e-05,
       5.24374957e-05, 5.29416290e-05, 5.25395594e-05, 5.25345340e-05,
       5.27847456e-05, 5.19925713e-05, 5.27561207e-05, 5.26861072e-05,
       5.25215358e-05, 5.23542377e-05, 5.24205504e-05, 5.20894094e-05,
       5.27775252e-05, 5.25651484e-05, 5.22221154e-05, 5.28711553e-05,
       5.25564873e-05, 5.21012408e-05, 5.30044813e-05, 5.28361825e-05,
       5.13447330e-05, 5.22164881e-05]), 'analog_power': array([1.17465585e-05, 1.19348250e-05, 1.23008853e-05, 1.15634605e-05,
       1.31435621e-05, 1.27060542e-05, 1.15766181e-05, 1.26167961e-05,
       1.23855745e-05, 1.26804249e-05, 1.25783420e-05, 1.18251313e-05,
       

In [5]:
# - Measure idle power (no evolution)
from time import sleep
import samna

# For XyloA3 power measurement is activated at the beginning of the evolve function and deactivated at the end
# to measure idle power, we need to activate power measurement explicitly
if xylo_board_name == 'XyloAudio3':
    modSamna._power_monitor.start_auto_power_measurement(20)

modSamna._power_buf.get_events()
sleep(5.)
power = modSamna._power_buf.get_events()

# - For XyloAudio 2
if xylo_board_name == 'XyloAudio2':
    power_idle = ([], [], [], [])
    for p in power:
        power_idle[p.channel].append(p.value)

    idle_power_per_channel = np.mean(np.stack(power_idle), axis = 1)
    
    channels = samna.xyloA2TestBoard.MeasurementChannels
    io_power = idle_power_per_channel[channels.Io]
    afe_core_power = idle_power_per_channel[channels.LogicAfe]
    afe_ldo_power = idle_power_per_channel[channels.IoAfe]
    snn_core_power = idle_power_per_channel[channels.Logic]
    
    print(f'XyloAudio 2\nAll IO:\t\t{io_power * 1e6:.1f} µW\nAFE core:\t{afe_core_power * 1e6:.1f} µW\nInternal LDO:\t{afe_ldo_power * 1e6:.1f} µW\nSNN core logic:\t{snn_core_power*1e6:.1f} µW')

# - For XyloAudio 3
elif xylo_board_name == 'XyloAudio3':
    # deactivate power measurement
    modSamna._power_monitor.stop_auto_power_measurement()
    
    power_idle = ([], [], [])
    for p in power:
        power_idle[p.channel].append(p.value)

    channels = samna.xyloAudio3.MeasurementChannels
    idle_power_per_channel = np.mean(np.stack(power_idle), axis = 1)
    io_power = idle_power_per_channel[channels.Io] 
    analog_power = idle_power_per_channel[channels.AnalogLogic] 
    digital_power = idle_power_per_channel[channels.DigitalLogic]
    print(f'XyloAudio 3\nAll IO:\t\t{io_power * 1e6:.1f} µW\nAFE core:\t{analog_power * 1e6:.1f} µW\nSNN core logic:\t{digital_power*1e6:.1f} µW')


XyloAudio 3
All IO:		39.7 µW
AFE core:	12.2 µW
SNN core logic:	577.6 µW


### About power measurement on XyloAudio 3

XyloAudio 3 power is sampled in three nets:
* IO power measures the power necessary to communicate with the outside world. 
* Analog power measures the power of the analog frontend and ADC.
* Digital power measures the power consumption of the digital logic (XyloAudio 3 internal digital power, excluding IO power)

The digital power consumption primarily depends on the clock speed, and the size of the network. The number of spikes generated per time step doesn't matter that much on power consumption.

XyloAudio 3 digital microphone is fed by an external power source and is not included in the Xylo power measurements.

The internal block on Xylo that receives the PDM signal and does the signal processing (Digital Front End) is powered by the same power rail as the SNN core. Those blocks are included in the power measurement.
And while all digital logic in XyloAudio 3 is powered with the same power supply, and thus the static power (without any clock) of the DFE and SNN core can't be distinguished, it is possible to distinguish the dynamic powers between DFE and SNN Core, by:
** feeding PDM_CLK externally, also feeding meaningful PDM data, disabling main clock -> we will get the DFE dynamic power
** don't activate PDM_IF, feeding spikes via SAER_I to SNN Core -> we will get the SNN core dynamic power

Also note that:

When XyloAudio 3 is powered on and no clocks are provided (disabling main clock as well via FPGA settings), the digital measured power is the total DFE + SNN Core static power (P0).
When deploying a network and feeding spikes via the SAER_I interface and running SNN core in parallel, the measured digital power (P1) is P0 + dynamic SNN core power -> P1 - P0 = dynamic SNN core power.
Note: P1 contains minor power from SAER_I processing, it is considered to be very small and can be ignored, also P1 is measured with SNN core in Accelerated mode.
When deploying a network and activating digital microphone path as normal, the measured power (P2) at VDDHD is P0 + dynamic SNN core power + dynamic DFE power -> P2 - P1 = dynamic DFE power.
Note: P2 is measured with SNN core in real-time mode.


Accelerated and real-time will differ in power consumption because accelerated mode will in most cases be in the active phase while real time mode will switch between active and idle.

### Hints on reducing power consumption

In [7]:
if xylo_board_name == 'XyloAudio2':
    help(xa_utils.set_xylo_core_clock_freq)
elif xylo_board_name == 'XyloAudio3':
    help(xa_utils.set_xylo_core_clock_freq)   


Help on function set_xylo_core_clock_freq in module rockpool.devices.xylo.syns65302.xa3_devkit_utils:

set_xylo_core_clock_freq(device: samna.xyloAudio3.XyloAudio3TestBoard, main_clock_frequency_MHz: float) -> None
    Set the internal core clock frequency used by Xylo

    Args:
        device (XyloAudio3HDK): A XyloAudio 3 device to configure
        main_clock_frequency_MHz (float): The main clock frequency of XyloAudio 3 in MHz



### Estimate the required master clock frequency for real-time operation

In [8]:
if xylo_board_name == 'XyloAudio2':
    from rockpool.devices.xylo.syns61201 import cycles_model, est_clock_freq
elif xylo_board_name == 'XyloAudio3':
    from rockpool.devices.xylo.syns65302 import cycles_model, est_clock_freq
help(cycles_model)

Help on function cycles_model in module rockpool.devices.xylo.syns63300.power_cycles_model:

cycles_model(config: Union[samna.xyloImu.configuration.XyloConfiguration, samna.xyloAudio3.configuration.XyloConfiguration], input_sp: Union[float, numpy.ndarray, torch.Tensor, array] = 1.0, hidden_sp: Union[float, numpy.ndarray, torch.Tensor, array] = 1.0, output_sp: Union[float, numpy.ndarray, torch.Tensor, array] = 1.0) -> float
    Calculate the average number of cycles required for a given network architecture

    This function contains a model which estimates the number of master clock cycles required for the Xylo SNN SYNS61202 and SYNS65302 inference cores to compute one time-step for a given chip configuration in ``config``. Use :py:func:`~.devices.xylo.syns61201.config_from_specification` to obtain a chip configuration, along with :py:meth:`.Module.as_graph` and :py:func:`~.devices.xylo.syns61201.mapper`, as described in the deployment tutorials for Xylo.

    By default the model pro

In [9]:
help(est_clock_freq)

Help on function est_clock_freq in module rockpool.devices.xylo.syns63300.power_cycles_model:

est_clock_freq(config: Union[samna.xyloImu.configuration.XyloConfiguration, samna.xyloAudio3.configuration.XyloConfiguration], dt: float, margin: float = 0.2)
    Estimate the required master clock frequency, to run a network in real-time.

    This function will perform a worst-case analysis, assuming that every input channel, every hidden neuron and every output neuron fire an event on each `dt`.
    An additional margin is included (Default: 20%), to guarantee that the model will run in real time at the suggested master clock frequency.
    Note: The clock frequency is returned in Hz.

    Args:
        config (Union[XyloIMUConfig, XyloA3Config]):  A Xylo configuration for which to estimate the required clock frequency
        dt (float): The required network `dt`, in seconds
        margin (float): The additional overhead safety margin to add to the estimation, as a fraction. Default: `0.

In [10]:
print(f"This network requires {cycles_model(config)} master clock cycles per network time-step.")
print(f"This network requires a master clock of {est_clock_freq(config, dt) / 1e6:.2f} MHz for real-time operation.")

This network requires 562.0 master clock cycles per network time-step.
This network requires a master clock of 0.67 MHz for real-time operation.
